<a href="https://colab.research.google.com/github/pnperl/Travel_Router/blob/main/Travel_Router.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================================
# LIVE INDIAN RAILWAY HOPPING ROUTE FINDER
# FULL WORKING GOOGLE COLAB CODE
# ============================================================

# FEATURES
# --------
# ✅ Direct + Hopping Routes
# ✅ Group Booking Optimization
# ✅ AC Preference Scoring
# ✅ Night Journey Preference
# ✅ Dynamic Route Generation
# ✅ Bus + Train Expandable
# ✅ Works WITHOUT paid APIs initially
#
# NOTE
# ----
# This version uses:
# - static realistic train dataset
# - expandable architecture
#
# You can later connect:
# - RailYatri
# - ConfirmTkt
# - RapidAPI
# - IRCTC partner APIs
#
# ============================================================

!pip -q install pandas tabulate networkx

import pandas as pd
import networkx as nx
from tabulate import tabulate
from datetime import datetime, timedelta

# ============================================================
# USER INPUTS
# ============================================================

SOURCE = "JAM"      # Jamnagar
DEST = "MMCT"       # Mumbai Central

GROUP_SIZE = 10

TRAVEL_DATE = "2026-05-12"

MAX_HOPS = 2

# Preferences
PREFER_AC = True
PREFER_NIGHT = True

# ============================================================
# REALISTIC TRAIN DATABASE
# ============================================================

# You can replace this later with LIVE APIs

TRAINS = [

    # ========================================================
    # DIRECT TRAINS
    # ========================================================

    {
        "train_no": "22946",
        "train_name": "Saurashtra Mail",
        "from": "JAM",
        "to": "DDR",
        "dep": "20:30",
        "arr": "10:30",
        "class": "3A",
        "availability": 8
    },

    {
        "train_no": "19218",
        "train_name": "Saurashtra Janta",
        "from": "JAM",
        "to": "BDTS",
        "dep": "21:45",
        "arr": "12:00",
        "class": "SL",
        "availability": 72
    },

    {
        "train_no": "22960",
        "train_name": "Humsafar Express",
        "from": "JAM",
        "to": "BDTS",
        "dep": "19:30",
        "arr": "09:30",
        "class": "3A",
        "availability": 12
    },

    # ========================================================
    # JAMNAGAR -> RAJKOT
    # ========================================================

    {
        "train_no": "09522",
        "train_name": "Intercity Express",
        "from": "JAM",
        "to": "RJT",
        "dep": "16:00",
        "arr": "18:30",
        "class": "2S",
        "availability": 120
    },

    # ========================================================
    # JAMNAGAR -> AHMEDABAD
    # ========================================================

    {
        "train_no": "19205",
        "train_name": "Saurashtra Express",
        "from": "JAM",
        "to": "ADI",
        "dep": "13:00",
        "arr": "19:00",
        "class": "CC",
        "availability": 45
    },

    {
        "train_no": "22946",
        "train_name": "Saurashtra Mail",
        "from": "JAM",
        "to": "ADI",
        "dep": "20:30",
        "arr": "04:30",
        "class": "SL",
        "availability": 58
    },

    # ========================================================
    # RAJKOT -> MUMBAI
    # ========================================================

    {
        "train_no": "22946",
        "train_name": "Saurashtra Mail",
        "from": "RJT",
        "to": "DDR",
        "dep": "21:00",
        "arr": "10:30",
        "class": "3A",
        "availability": 9
    },

    {
        "train_no": "19218",
        "train_name": "Saurashtra Janta",
        "from": "RJT",
        "to": "BDTS",
        "dep": "22:30",
        "arr": "12:00",
        "class": "SL",
        "availability": 62
    },

    # ========================================================
    # AHMEDABAD -> MUMBAI
    # ========================================================

    {
        "train_no": "12902",
        "train_name": "Gujarat Mail",
        "from": "ADI",
        "to": "MMCT",
        "dep": "22:00",
        "arr": "06:00",
        "class": "3A",
        "availability": 28
    },

    {
        "train_no": "22954",
        "train_name": "Lok Shakti Express",
        "from": "ADI",
        "to": "BDTS",
        "dep": "20:30",
        "arr": "07:00",
        "class": "3A",
        "availability": 14
    },

    {
        "train_no": "12268",
        "train_name": "Garib Rath",
        "from": "ADI",
        "to": "BDTS",
        "dep": "23:00",
        "arr": "08:00",
        "class": "3A",
        "availability": 36
    },

    {
        "train_no": "12932",
        "train_name": "AC Double Decker",
        "from": "ADI",
        "to": "MMCT",
        "dep": "15:00",
        "arr": "22:00",
        "class": "CC",
        "availability": 40
    }
]

df = pd.DataFrame(TRAINS)

# ============================================================
# TIME HELPERS
# ============================================================

def to_datetime(t):

    return datetime.strptime(t, "%H:%M")

def calculate_layover(arrival, departure):

    arr = to_datetime(arrival)
    dep = to_datetime(departure)

    if dep < arr:
        dep += timedelta(days=1)

    return int((dep - arr).seconds / 60)

def is_night_train(dep, arr):

    dep_hr = int(dep.split(":")[0])
    arr_hr = int(arr.split(":")[0])

    return dep_hr >= 19 or arr_hr <= 8

# ============================================================
# SCORING ENGINE
# ============================================================

def calculate_score(route):

    score = 0

    # ========================================================
    # Availability
    # ========================================================

    min_avail = min([x["availability"] for x in route])

    score += min(min_avail, 50)

    if min_avail >= GROUP_SIZE:
        score += 100

    # ========================================================
    # AC Preference
    # ========================================================

    if PREFER_AC:

        for leg in route:

            if leg["class"] == "3A":
                score += 40

            elif leg["class"] in ["CC", "2A"]:
                score += 25

    # ========================================================
    # Night Preference
    # ========================================================

    if PREFER_NIGHT:

        for leg in route:

            if is_night_train(leg["dep"], leg["arr"]):
                score += 20

    # ========================================================
    # Penalize daytime sleeper
    # ========================================================

    for leg in route:

        dep_hr = int(leg["dep"].split(":")[0])

        if (
            leg["class"] == "SL"
            and 10 <= dep_hr <= 17
        ):
            score -= 20

    return score

# ============================================================
# DIRECT ROUTES
# ============================================================

def find_direct_routes():

    direct = df[
        (df["from"] == SOURCE)
        &
        (
            df["to"].isin(
                ["MMCT", "BDTS", "DDR"]
            )
        )
    ]

    results = []

    for _, row in direct.iterrows():

        route = [row]

        score = calculate_score(route)

        results.append({

            "type": "DIRECT",

            "route":
                f"{SOURCE} -> {row['to']}",

            "journey":

                f"{row['train_name']} "
                f"({row['dep']} - {row['arr']}) "
                f"{row['class']}",

            "availability":
                row["availability"],

            "score": score,

            "recommended":
                "YES"
                if row["availability"] >= GROUP_SIZE
                else "PARTIAL"
        })

    return results

# ============================================================
# HOPPING ROUTES
# ============================================================

def find_hopping_routes():

    results = []

    first_legs = df[df["from"] == SOURCE]

    for _, leg1 in first_legs.iterrows():

        hub = leg1["to"]

        second_legs = df[
            (df["from"] == hub)
            &
            (
                df["to"].isin(
                    ["MMCT", "BDTS", "DDR"]
                )
            )
        ]

        for _, leg2 in second_legs.iterrows():

            layover = calculate_layover(
                leg1["arr"],
                leg2["dep"]
            )

            # Minimum 45 min layover
            if layover < 45:
                continue

            route = [leg1, leg2]

            score = calculate_score(route)

            min_avail = min(
                leg1["availability"],
                leg2["availability"]
            )

            results.append({

                "type": "HOPPING",

                "route":
                    f"{SOURCE} -> "
                    f"{hub} -> Mumbai",

                "journey":

                    f"{leg1['train_name']} "
                    f"({leg1['dep']}-{leg1['arr']}) "
                    f"{leg1['class']}\n"
                    f"↓ Layover {layover} mins ↓\n"
                    f"{leg2['train_name']} "
                    f"({leg2['dep']}-{leg2['arr']}) "
                    f"{leg2['class']}",

                "availability":
                    min_avail,

                "score": score,

                "recommended":
                    "YES"
                    if min_avail >= GROUP_SIZE
                    else "PARTIAL"
            })

    return results

# ============================================================
# RUN ENGINE
# ============================================================

direct_routes = find_direct_routes()

hopping_routes = find_hopping_routes()

all_routes = direct_routes + hopping_routes

results_df = pd.DataFrame(all_routes)

results_df = results_df.sort_values(
    by="score",
    ascending=False
)

# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\n")
print("=" * 80)
print("BEST ROUTE OPTIONS")
print("=" * 80)
print("\n")

if len(results_df) == 0:

    print("No routes found")

else:

    print(tabulate(

        results_df[
            [
                "type",
                "route",
                "journey",
                "availability",
                "recommended",
                "score"
            ]
        ],

        headers="keys",
        tablefmt="fancy_grid",
        showindex=False

    ))

# ============================================================
# BEST RECOMMENDATION
# ============================================================

print("\n")
print("=" * 80)
print("TOP RECOMMENDED OPTION")
print("=" * 80)

best = results_df.iloc[0]

print("\nROUTE:")
print(best["route"])

print("\nJOURNEY:")
print(best["journey"])

print("\nAVAILABLE SEATS:")
print(best["availability"])

print("\nBOOKING FEASIBILITY:")
print(best["recommended"])

print("\nCOMFORT SCORE:")
print(best["score"])

# ============================================================
# OPTIONAL:
# SAVE RESULTS
# ============================================================

results_df.to_csv(
    "railway_route_results.csv",
    index=False
)

print("\nCSV SAVED:")
print("railway_route_results.csv")



BEST ROUTE OPTIONS


╒═════════╤══════════════════════╤═════════════════════════════════════╤════════════════╤═══════════════╤═════════╕
│ type    │ route                │ journey                             │   availability │ recommended   │   score │
╞═════════╪══════════════════════╪═════════════════════════════════════╪════════════════╪═══════════════╪═════════╡
│ HOPPING │ JAM -> ADI -> Mumbai │ Saurashtra Express (13:00-19:00) CC │             36 │ YES           │     221 │
│         │                      │ ↓ Layover 240 mins ↓                │                │               │         │
│         │                      │ Garib Rath (23:00-08:00) 3A         │                │               │         │
├─────────┼──────────────────────┼─────────────────────────────────────┼────────────────┼───────────────┼─────────┤
│ HOPPING │ JAM -> ADI -> Mumbai │ Saurashtra Mail (20:30-04:30) SL    │             36 │ YES           │     216 │
│         │                      │ ↓ Layover 1110